# Sesión AWS prestada y ADA opcional · 2.1.4

Input → sesión preparada, región y modelo. Proceso → runtime valida, Agent compila y ADA inspecciona sin resolver credenciales. Output → configuración segura. Este es un control offline de configuración, no una validación IAM. No se instala nada ni se llama a un servicio. Requiere el SDK y el complemento ADA locales. Todas las declaraciones están visibles.

In [2]:
"""Borrow an AWS session: configuration only, not IAM or ADA certification."""

import json

import boto3
import agentic_systems as toolkit
import agentic_systems_ada as ada


# This explicit offline fixture avoids looking up credentials or metadata.
# In ADA, use the session supplied by your platform instead of fixture keys.
session = boto3.Session(
    region_name="us-east-2",
    aws_access_key_id="offline-fixture",
    aws_secret_access_key="offline-fixture",
)
connection = toolkit.runtime(
    provider="bedrock-runtime", model="qwen-model-enabled-in-your-account",
    session=session,
)
agent = toolkit.Agent("aws_worker", runtime=connection)
program = agent.compile()


def run():
    # compile prepares; it does not create clients or invoke a model.
    assert connection.region == "us-east-2"
    assert "session" not in connection.model_dump()
    report = ada.diagnose(session=session)
    assert not report["services_checked"] and not report["credentials_resolved"]
    return {
        "input": "Borrowed boto3 Session, model and region",
        "process": "Validate connection -> compile Agent -> inspect configuration without services",
        "output": {"runtime": connection.inspect(), "diagnostic": report},
        "validation_scope": "configuration-offline-passed",
        "pending": ["real AWS IAM", "real ADA IAM", "model access and business acceptance"],
        "inference_requests": 0,
    }

In [3]:
report = run()
print(json.dumps(report, ensure_ascii=False, indent=2))
assert report["validation_scope"] == "configuration-offline-passed"

{
  "input": "Borrowed boto3 Session, model and region",
  "process": "Validate connection -> compile Agent -> inspect configuration without services",
  "output": {
    "runtime": {
      "provider": "bedrock-runtime",
      "model": "qwen-model-enabled-in-your-account",
      "region": "us-east-2",
      "endpoint": null,
      "api_key_configured": false,
      "session_configured": true,
      "authentication_mode": "aws-session-iam",
      "timeout_s": 60.0,
      "max_retries": 0,
      "max_tool_calls": 5,
      "max_turns": 6,
      "max_tokens": null,
      "backoff_s": 0.0,
      "max_concurrency": 1
    },
    "diagnostic": {
      "version": "2.1.4.dev8",
      "scope": "configuration-only",
      "session_supplied": true,
      "session_region": "us-east-2",
      "environment_present": {
        "AWS_ACCESS_KEY_ID": false,
        "AWS_SECRET_ACCESS_KEY": false,
        "AWS_SESSION_TOKEN": false,
        "AWS_SECURITY_TOKEN": false,
        "AWS_PROFILE": false,
        

En tu entorno sustituye únicamente la sesión fixture por la sesión preparada por la plataforma. No pegues sus claves en el notebook. API key y sesión IAM no se mezclan; los conflictos se rechazan. El recurso se conserva fuera del estado y los resultados. El uso externo concurrente es responsabilidad del propietario. AWS IAM/ADA reales siguen pendientes. [Límites y ejecución](README.md).